# RAG01｜GitHub 自動讀取 → 建立 FAISS 向量資料庫

**主題：金門酒廠 2022 永續報告書**。這份 Notebook 沿用原本的 RAG01 核心流程：文字切塊 → SentenceTransformer Embedding → FAISS → `vector_embedding.zip`。

1. 把 `rag-data.zip` 上傳至**公開** GitHub Repository 的根目錄。
2. 在「② 設定 GitHub」填寫你的使用者名稱；Repo 名稱預設 `kinmen-rag-homework`。
3. 在 Colab 依序執行全部程式。**第一次需要連網下載免費模型**。
4. 最後下載 `vector_embedding.zip`，再上傳到相同的 GitHub Repository，供 RAG02 自動讀取。

注意：GitHub 用來放資料，**不會自行執行 Python 或 AI 模型**。向量化在 Colab 完成。


## ① 安裝套件（第一次執行需要網路）

使用開源多語 Embedding 模型 `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`，不需要 API Key，也不需要付費。

In [ ]:
!pip -q install "sentence-transformers>=3.0" "faiss-cpu>=1.8" "numpy>=1.24"

## ② 設定 GitHub 帳號，直接從公開 Repository 自動下載 ZIP

只需修改下面的 `GITHUB_USER`。GitHub 需有 `rag-data.zip`，並且 Repo 設為 **Public**。下載失敗時會提供手動上傳的備援。


In [ ]:
from pathlib import Path
import json, io, zipfile, hashlib, tempfile
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
# ========= 只需要修改這個設定區 =========
GITHUB_USER = "請填入你的GitHub帳號"
REPO_NAME = "kinmen-rag-homework"
BRANCH = "main"
# ====================================

GITHUB_USER = GITHUB_USER.strip()
REPO_NAME = REPO_NAME.strip()
BRANCH = BRANCH.strip()
if not GITHUB_USER or GITHUB_USER == "請填入你的GitHub帳號":
    raise ValueError("請先把 GITHUB_USER 改成自己的 GitHub 使用者名稱，再執行本格。")
GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}"
print("GitHub 資料位置：", GITHUB_RAW_BASE)

import requests
from google.colab import files
import io, zipfile

def read_zip_from_github_or_upload(filename):
    # GitHub 公開儲存庫：每次執行 Colab 都能取得檔案，不依賴 Google Drive
    url = f"{GITHUB_RAW_BASE}/{filename}"
    print("嘗試下載：", url)
    try:
        resp = requests.get(url, timeout=60, headers={"User-Agent": "Colab-RAG-Homework"})
        resp.raise_for_status()
        data = resp.content
        if not zipfile.is_zipfile(io.BytesIO(data)):
            raise ValueError("網址有回傳資料，但內容不是有效 ZIP。")
        print(f"GitHub 下載成功：{filename} ({len(data):,} bytes)")
        return data
    except (requests.RequestException, ValueError) as error:
        print(f"GitHub 自動下載失敗：{error}")
        print("請確認 Repo 設為 Public、檔名與大小寫正確，或改用手動上傳。")
        uploaded = files.upload()
        if filename not in uploaded:
            raise ValueError(f"請上傳名稱正確的 {filename}，不要上傳其他 ZIP。")
        data = uploaded[filename]
        if not zipfile.is_zipfile(io.BytesIO(data)):
            raise ValueError(f"上傳的 {filename} 並非有效 ZIP。")
        return data

DATA_FILENAME = 'rag-data.zip'
OUTPUT_FILENAME = 'vector_embedding.zip'
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'


## ③ 從 GitHub 自動讀取 `rag-data.zip`

這份 ZIP 內含從原始 2022 年永續報告書整理的 `.txt` 以及 `manifest.json`，資料保留來源頁碼。


In [ ]:
data_zip_bytes = read_zip_from_github_or_upload(DATA_FILENAME)
source_sha256 = hashlib.sha256(data_zip_bytes).hexdigest()
with zipfile.ZipFile(io.BytesIO(data_zip_bytes), 'r') as archive:
    names = archive.namelist()
    txt_names = sorted(n for n in names if n.lower().endswith('.txt') and not n.startswith('__MACOSX/'))
    if not txt_names:
        raise ValueError('ZIP 裡沒有 .txt 檔案。')
    if len(txt_names) > 5000:
        raise ValueError('檔案數量異常。')
    for n in txt_names:
        if n.startswith('/') or '..' in Path(n).parts or archive.getinfo(n).file_size > 3_000_000:
            raise ValueError(f'不安全或過大的檔案：{n}')
    manifest = json.loads(archive.read('manifest.json').decode('utf-8')) if 'manifest.json' in names else {}
    by_file = {x['file']: x for x in manifest.get('entries', [])}
    documents = []
    for idx, name in enumerate(txt_names):
        content = archive.read(name).decode('utf-8-sig').strip()
        if not content: continue
        m = by_file.get(name,{})
        documents.append({
            'id': m.get('id',str(idx)),
            'text': content,
            'file': name,
            'title': m.get('title','自訂文件'),
            'chapter': m.get('chapter','未分類'),
            'report_page': m.get('report_page'),
            'pdf_page': m.get('pdf_page'),
            'year': m.get('year'),
        })
print(f'讀取成功：{len(documents)} 段文字，來源 SHA-256：{source_sha256[:16]}...')
print('範例文件：', documents[0]['file'])
print(documents[0]['text'][:250])

## ④ 建立真正的 Embedding 與 FAISS 向量索引

將每段文字轉換成語意向量（不是單純把文字存進 ZIP），再使用正規化與內積建立 cosine similarity 搜尋。

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)
texts_to_embed = [d['text'] for d in documents]
embeddings = model.encode(
    texts_to_embed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype('float32')

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
index_doc_ids = [d['id'] for d in documents]
print('FAISS 向量維度：', index.d)
print('索引文件數量：', index.ntotal)

## ⑤ 匯出符合老師格式的 `vector_embedding.zip`

根目錄恰好有 `index.faiss`、`documents.json`、`index_doc_ids.json`、`config.json`。執行後 Colab 會下載 ZIP 到你電腦；請**將這個 ZIP 手動上傳 GitHub**，RAG02 才能直接下載。


In [ ]:
from datetime import datetime, timezone
from google.colab import files
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    faiss.write_index(index, str(base / 'index.faiss'))
    (base / 'documents.json').write_text(json.dumps(documents,ensure_ascii=False,indent=2),encoding='utf-8')
    (base / 'index_doc_ids.json').write_text(json.dumps(index_doc_ids,ensure_ascii=False,indent=2),encoding='utf-8')
    digest = {f:hashlib.sha256((base / f).read_bytes()).hexdigest() for f in ['index.faiss','documents.json','index_doc_ids.json']}
    config = {
        'embedding_model': EMBEDDING_MODEL,
        'embedding_dimension': int(index.d),
        'similarity': 'cosine_similarity_via_normalized_inner_product',
        'normalized': True,
        'documents_count': len(documents),
        'source_archive_sha256': source_sha256,
        'source_title': '金門酒廠實業股份有限公司 2022 年永續報告書',
        'file_sha256': digest,
        'created_utc': datetime.now(timezone.utc).isoformat(),
    }
    (base / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2),encoding='utf-8')
    output_path = Path('/content') / OUTPUT_FILENAME
    with zipfile.ZipFile(output_path,'w',zipfile.ZIP_DEFLATED) as z:
        for filename in ['index.faiss','documents.json','index_doc_ids.json','config.json']:
            z.write(base / filename,arcname=filename)
    print('完成：',output_path, '大小（KB）：',round(output_path.stat().st_size/1024,1))
    with zipfile.ZipFile(output_path) as check:
        assert set(check.namelist()) == {'index.faiss','documents.json','index_doc_ids.json','config.json'}

print('現在請把下載得到的 vector_embedding.zip 上傳到 GitHub Repo 根目錄。')
files.download(str(output_path))

## ⑥ 實測檢索（建議截圖放作業）

In [ ]:
question = '金門酒廠2022年的合併營收是多少？'
q = model.encode([question],normalize_embeddings=True,convert_to_numpy=True).astype('float32')
scores, ids = index.search(q, 3)
for rank,(score,idx) in enumerate(zip(scores[0],ids[0]),1):
    if idx < 0: continue
    d=documents[int(idx)]
    print(f'第{rank}筆 | 相似度 {score:.3f} | 報告書第{d["report_page"]}頁 | {d["chapter"]}')
    print(d['text'][:360], '\n')

**完成 RAG01 後：** 在 GitHub 的 Repository 根目錄按 `Add file → Upload files`，加入剛剛下載的 **`vector_embedding.zip`** 並提交（Commit）。接著開啟 RAG02。

若不想每次建立向量，**保留此 ZIP** 即可；GitHub 修改後需要重新執行 RAG02 才會載入新版資料。
